In [1]:
import os
from pathlib import Path
import shutil

import teehr

teehr.__version__

'0.6.0dev2'

In [2]:
from teehr.evaluation.spark_session_utils import create_spark_session

In [3]:
import boto3
session = boto3.Session()
credentials = session.get_credentials()
aws_access_key_id = credentials.access_key
aws_secret_access_key = credentials.secret_key
aws_region = session.region_name
os.environ["AWS_ACCESS_KEY_ID"] = aws_access_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret_access_key
os.environ["AWS_REGION"] = aws_region

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


In [4]:
spark = create_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS credentials from environment variables
INFO:teehr.evaluation.spark_session_utils:   - Using long-term credentials
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:Spark session created for TEEHR Evaluation.
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [5]:
%%time
dir_path = "/data/temp_warehouse"

ev = teehr.Evaluation(
    spark=spark,
    dir_path=dir_path,
    check_evaluation_version=False,
)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to local.


CPU times: user 3.54 ms, sys: 202 μs, total: 3.74 ms
Wall time: 102 ms


In [6]:
ev.set_active_catalog("remote")

ev.active_catalog

INFO:teehr.evaluation.evaluation:Active catalog set to remote.


RemoteCatalog(warehouse_dir='s3://dev-teehr-iceberg-warehouse/', catalog_name='iceberg', namespace_name='teehr', catalog_type='rest', catalog_uri='http://iceberg-rest:8181')

#### Attributes

In [14]:
existing_sdf = ev.attributes.to_sdf()

INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.attributes.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.attributes.


In [18]:
existing_sdf.show()

+-------------------+--------------------+-----------+
|               name|         description|       type|
+-------------------+--------------------+-----------+
|       aggecoregion|Aaggregated level...|categorical|
|           bfi_mean|Mean baseflow ind...| continuous|
|             county|              County|categorical|
|dam_dist_nearest_km|Index of anthropo...| continuous|
|  disturbance_index|Index of anthropo...| continuous|
|  drainage_area_km2|Drainage area in ...| continuous|
|       ecoregion_L2|EPA Level II ecor...|categorical|
|        elev_mean_m|Mean watershed el...| continuous|
|      gagesII_class|GAGES II class as...|categorical|
|     horton_percent|Percent horton ov...| continuous|
|              huc10|HUC10 containing ...|categorical|
|              huc12|HUC12 containing ...|categorical|
|               huc2|HUC2 containing t...|categorical|
|               huc4|HUC4 containing t...|categorical|
|               huc6|HUC6 containing t...|categorical|
|         

In [9]:
options = {
    "header": "true",
    "ignoreMissingFiles": "true"
}

tbl = ev.attributes()
schema = tbl.schema_func().to_structtype()
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e1_camels_daily_streamflow/dataset/attributes/"

sdf = spark.read.format("csv").options(**options).load(s3_dirpath, schema=schema) 

In [20]:
sdf.join(existing_sdf, "name", "left_anti").show()

+--------------------+-----------+--------------------+
|                name|       type|         description|
+--------------------+-----------+--------------------+
|         2_year_flow| continuous|Two year flow rat...|
|           frac_snow|categorical|           frac_snow|
|         high_q_freq|categorical|         high_q_freq|
|      dom_land_cover|categorical|      dom_land_cover|
|                  q5|categorical|                  q5|
|       p_seasonality|categorical|       p_seasonality|
|      baseflow_index|categorical|      baseflow_index|
|         zero_q_freq|categorical|         zero_q_freq|
|     NID_dam_lengths|categorical|     NID_dam_lengths|
|          slope_mean|categorical|          slope_mean|
|                 q95|categorical|                 q95|
|       soil_porosity|categorical|       soil_porosity|
|              p_mean|categorical|              p_mean|
|           slope_fdc|categorical|           slope_fdc|
| dom_land_cover_frac|categorical| dom_land_cove

In [9]:
# ev.write.to_warehouse(table_name="attributes", source_data=sdf, write_mode="upsert")

#### Locations

In [22]:
tbl = ev.locations()
existing_sdf = tbl.to_sdf()
schema = tbl.schema_func().to_structtype()
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e1_camels_daily_streamflow/dataset/locations/"

sdf = spark.read.format("parquet").options(**options).load(s3_dirpath, schema=schema) 

INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.


In [23]:
sdf.join(existing_sdf, "name", "left_anti").show()

+----+---+--------+
|name| id|geometry|
+----+---+--------+
+----+---+--------+



In [15]:
# ev.write.to_warehouse(table_name="locations", source_data=sdf, write_mode="upsert")

#### Location attributes

In [24]:
tbl = ev.location_attributes()
schema = tbl.schema_func().to_structtype()
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e1_camels_daily_streamflow/dataset/location_attributes/"

sdf = spark.read.format("parquet").options(**options).load(s3_dirpath, schema=schema) 

In [17]:
# ev.write.to_warehouse(table_name="location_attributes", source_data=sdf, write_mode="upsert")

#### Location crosswalks

In [31]:
tbl = ev.location_crosswalks()
existing_sdf = tbl.to_sdf()
schema = tbl.schema_func().to_structtype()
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e1_camels_daily_streamflow/dataset/location_crosswalks/"

sdf = spark.read.format("parquet").options(**options).load(s3_dirpath, schema=schema) 

INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.location_crosswalks.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.location_crosswalks.


In [32]:
sdf.join(existing_sdf, "primary_location_id", "left_anti").show()

+-------------------+---------------------+
|primary_location_id|secondary_location_id|
+-------------------+---------------------+
+-------------------+---------------------+



In [33]:
sdf.join(existing_sdf, "secondary_location_id", "left_anti").show()

+---------------------+-------------------+
|secondary_location_id|primary_location_id|
+---------------------+-------------------+
|   huc12-010200030704|      usgs-01030500|
|   huc12-040301120207|      usgs-04057510|
|   huc12-180400100607|      usgs-11299600|
|   huc12-111101030501|      usgs-07195800|
|   huc12-111101030709|      usgs-07197000|
|   huc12-111101030701|      usgs-07196900|
|   huc12-050702010506|      usgs-03213700|
|   huc12-051002030606|      usgs-03281500|
|   huc12-111401050101|      usgs-07335700|
|   huc12-010100030403|      usgs-01013500|
|   huc12-110200010205|      usgs-07083000|
|   huc12-111403060208|      usgs-07346045|
|   huc12-030701011607|      usgs-02221525|
|   huc12-160300070207|      usgs-10234500|
|   huc12-140100010401|      usgs-09035900|
|   huc12-120301030201|      usgs-08050800|
|   huc12-140100020203|      usgs-09047700|
|   huc12-140100030103|      usgs-09066300|
|   huc12-140100030102|      usgs-09066200|
|   huc12-140100030101|      usg

In [19]:
# ev.write.to_warehouse(table_name="location_crosswalks", source_data=sdf, write_mode="upsert")

#### Units

In [34]:
tbl = ev.units()
existing_sdf = tbl.to_sdf()
schema = tbl.schema_func().to_structtype()
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e1_camels_daily_streamflow/dataset/units/"

sdf = spark.read.format("csv").options(**options).load(s3_dirpath, schema=schema) 

sdf.join(existing_sdf, "name", "left_anti").show()

INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.units.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.units.


+----+---------+
|name|long_name|
+----+---------+
+----+---------+



In [ ]:
# ev.write.to_warehouse(table_name="units", source_data=sdf, write_mode="upsert")

#### Variables

In [36]:
tbl = ev.variables()
existing_sdf = tbl.to_sdf()
schema = tbl.schema_func().to_structtype()
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e1_camels_daily_streamflow/dataset/variables/"

sdf = spark.read.format("csv").options(**options).load(s3_dirpath, schema=schema) 

sdf.join(existing_sdf, "name", "left_anti").show()

INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.variables.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.variables.


+----+---------+
|name|long_name|
+----+---------+
+----+---------+



In [ ]:
# ev.write.to_warehouse(table_name="variables", source_data=sdf, write_mode="upsert")

#### Configurations

In [38]:
tbl = ev.configurations()
existing_sdf = tbl.to_sdf()
schema = tbl.schema_func().to_structtype()
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e1_camels_daily_streamflow/dataset/configurations/"

sdf = spark.read.format("csv").options(**options).load(s3_dirpath, schema=schema) 

sdf.join(existing_sdf, "name", "left_anti").show()

INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.


+-------------------+---------+-------------------+
|               name|     type|        description|
+-------------------+---------+-------------------+
|marrmot_37_hbv_obj1|secondary|Marrmot 37_hbv_obj1|
|   camels_daymet_05|secondary|   camels_daymet_05|
+-------------------+---------+-------------------+



In [ ]:
# ev.write.to_warehouse(table_name="configurations", source_data=sdf, write_mode="upsert")

#### Primary Timeseries

In [39]:
tbl = ev.primary_timeseries()
schema = tbl.schema_func().to_structtype()
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e1_camels_daily_streamflow/dataset/primary_timeseries/"

sdf = spark.read.format("parquet").options(**options).load(s3_dirpath, schema=schema) 
sdf.show()

+--------------+-------------------+---------+---------+-------------+------------------+--------------------+
|reference_time|         value_time|    value|unit_name|  location_id|configuration_name|       variable_name|
+--------------+-------------------+---------+---------+-------------+------------------+--------------------+
|          NULL|1990-10-01 00:00:00|23.419521|    m^3/s|usgs-01013500| usgs_observations|streamflow_daily_...|
|          NULL|1990-10-02 00:00:00| 25.04979|    m^3/s|usgs-01013500| usgs_observations|streamflow_daily_...|
|          NULL|1990-10-03 00:00:00|26.965897|    m^3/s|usgs-01013500| usgs_observations|streamflow_daily_...|
|          NULL|1990-10-04 00:00:00| 28.62951|    m^3/s|usgs-01013500| usgs_observations|streamflow_daily_...|
|          NULL|1990-10-05 00:00:00|30.841764|    m^3/s|usgs-01013500| usgs_observations|streamflow_daily_...|
|          NULL|1990-10-06 00:00:00|  31.4199|    m^3/s|usgs-01013500| usgs_observations|streamflow_daily_...|
|

In [40]:
ev.write.to_warehouse(table_name="primary_timeseries", source_data=sdf, write_mode="upsert")

#### Secondary Timeseries

In [41]:
%%time
tbl = ev.secondary_timeseries()
schema = tbl.schema_func().to_structtype()
s3_dirpath = "s3a://ciroh-rti-public-data/teehr-data-warehouse/v0_4_evaluations/e1_camels_daily_streamflow/dataset/secondary_timeseries/"
sdf = spark.read.format("parquet").options(**options).load(s3_dirpath, schema=schema) 

CPU times: user 5.76 ms, sys: 0 ns, total: 5.76 ms
Wall time: 142 ms


In [50]:
sdf = sdf.where("configuration_name = 'nwm30_retrospective'")

In [51]:
ev.write.to_warehouse(table_name="secondary_timeseries", source_data=sdf, write_mode="upsert")

In [27]:
ev.secondary_timeseries.to_sdf().count()

INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.


3061335120

In [28]:
ev.secondary_timeseries.distinct_values("configuration_name")

INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.


['nwm30_retrospective']